In [3]:
import pandas as pd

df = pd.read_csv("harvard_orcid_complete_20251020_current_harvard_detail_with_names.csv.gz")

print(df.shape)
print(df.columns)
print(df.head())

(406882, 9)
Index(['item', 'content', 'ORCID', 'full_name', 'given_name', 'family_name',
       'credit_name', 'status_code', 'error'],
      dtype='object')
         item                                            content  \
0  Employment  Role: Ph.D. | Department: Organismic and Evolu...   
1   Education  Degree: B.A. | Institution: Harvard University...   
2        Work  Title: Computational analysis of fish-foil pai...   
3        Work  Title: Fish locomotor variation: connecting en...   
4        Work  Title: Quantifying the denticle multiverse: a ...   

                 ORCID      full_name given_name family_name credit_name  \
0  0000-0003-0731-286X  George Lauder     George      Lauder         NaN   
1  0000-0003-0731-286X  George Lauder     George      Lauder         NaN   
2  0000-0003-0731-286X  George Lauder     George      Lauder         NaN   
3  0000-0003-0731-286X  George Lauder     George      Lauder         NaN   
4  0000-0003-0731-286X  George Lauder     George     

C:\Users\ziw857\AppData\Local\Temp\ipykernel_15172\2217216761.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("harvard_orcid_complete_20251020_current_harvard_detail_with_names.csv.gz")


In [2]:
import time
import pandas as pd
import requests
from pathlib import Path

input_file = Path("harvard_orcid_complete_20251020_current_harvard_detail.csv.gz")
output_file = Path("harvard_orcid_complete_20251020_current_harvard_detail_with_names.csv.gz")
cache_file = Path("orcid_name_cache.csv")

df = pd.read_csv(input_file)

orcids = (
    df["ORCID"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

print("Rows:", len(df))
print("Unique ORCIDs:", len(orcids))

# Reuse previous results if the script is interrupted/restarted
if cache_file.exists():
    cache = pd.read_csv(cache_file)
    done = set(cache["ORCID"].astype(str))
    rows = cache.to_dict("records")
else:
    done = set()
    rows = []

headers = {
    "Accept": "application/json",
    "User-Agent": "Harvard ORCID name enrichment script"
}

for i, orcid in enumerate(orcids, start=1):
    if orcid in done:
        continue

    url = f"https://pub.orcid.org/v3.0/{orcid}/person"

    given_name = None
    family_name = None
    credit_name = None
    full_name = None
    status_code = None
    error = None

    try:
        r = requests.get(url, headers=headers, timeout=30)
        status_code = r.status_code

        if r.status_code == 200:
            data = r.json()
            name = data.get("name") or {}

            given_name = ((name.get("given-names") or {}).get("value"))
            family_name = ((name.get("family-name") or {}).get("value"))
            credit_name = ((name.get("credit-name") or {}).get("value"))

            if credit_name:
                full_name = credit_name
            else:
                parts = [x for x in [given_name, family_name] if x]
                full_name = " ".join(parts) if parts else None

        elif r.status_code == 404:
            error = "not_found"
        elif r.status_code == 429:
            error = "rate_limited"
            print("Rate limited. Sleeping 60 seconds...")
            time.sleep(60)
        else:
            error = f"http_{r.status_code}"

    except Exception as e:
        error = str(e)

    rows.append({
        "ORCID": orcid,
        "given_name": given_name,
        "family_name": family_name,
        "credit_name": credit_name,
        "full_name": full_name,
        "status_code": status_code,
        "error": error,
    })

    # Save cache frequently
    if i % 100 == 0:
        pd.DataFrame(rows).to_csv(cache_file, index=False)
        print(f"Processed {i}/{len(orcids)}")

    # Be polite to ORCID API
    time.sleep(0.2)

# Final cache save
names = pd.DataFrame(rows)
names.to_csv(cache_file, index=False)

# Merge names back to original long-detail file
df2 = df.merge(
    names[["ORCID", "full_name", "given_name", "family_name", "credit_name", "status_code", "error"]],
    on="ORCID",
    how="left"
)

df2.to_csv(output_file, index=False, compression="gzip")

print("Done.")
print("Output:", output_file)
print("Matched names:", df2["full_name"].notna().sum())
print("Unique ORCIDs with names:", names["full_name"].notna().sum())

Rows: 406882
Unique ORCIDs: 15863
Processed 100/15863
Processed 200/15863
Processed 300/15863
Processed 400/15863
Processed 500/15863
Processed 600/15863
Processed 700/15863
Processed 800/15863
Processed 900/15863
Processed 1000/15863
Processed 1100/15863
Processed 1200/15863
Processed 1300/15863
Processed 1400/15863
Processed 1500/15863
Processed 1600/15863
Processed 1700/15863
Processed 1800/15863
Processed 1900/15863
Processed 2000/15863
Processed 2100/15863
Processed 2200/15863
Processed 2300/15863
Processed 2400/15863
Processed 2500/15863
Processed 2600/15863
Processed 2700/15863
Processed 2800/15863
Processed 2900/15863
Processed 3000/15863
Processed 3100/15863
Processed 3200/15863
Processed 3300/15863
Processed 3400/15863
Processed 3500/15863
Processed 3600/15863
Processed 3700/15863
Processed 3800/15863
Processed 3900/15863
Processed 4000/15863
Processed 4100/15863
Processed 4200/15863
Processed 4300/15863
Processed 4400/15863
Processed 4500/15863
Processed 4600/15863
Processed

In [4]:
import pandas as pd

input_file = "harvard_orcid_complete_20251020_current_harvard_detail_with_names.csv.gz"
output_file = "harvard_orcid_unique_names.csv"

cols = [
    "ORCID",
    "full_name",
    "given_name",
    "family_name",
    "credit_name",
    "status_code",
    "error",
]

df = pd.read_csv(input_file, usecols=cols)

unique_names = (
    df[cols]
    .drop_duplicates(subset=["ORCID"])
    .sort_values("ORCID")
    .reset_index(drop=True)
)

unique_names.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(unique_names))
print(unique_names.head())

Saved: harvard_orcid_unique_names.csv
Rows: 15863
                 ORCID            full_name given_name   family_name  \
0  0000-0001-5002-7605     Thomas Del Prete     Thomas     Del Prete   
1  0000-0001-5006-2873  Xavier Roberts-Gaal     Xavier  Roberts-Gaal   
2  0000-0001-5006-3067  Carolyn M. Boudreau    Carolyn      Boudreau   
3  0000-0001-5007-193X             Rui Wang        Rui          Wang   
4  0000-0001-5010-2097         ZUI-SHEN YEN   ZUI-SHEN           YEN   

           credit_name  status_code error  
0                  NaN          200   NaN  
1                  NaN          200   NaN  
2  Carolyn M. Boudreau          200   NaN  
3                  NaN          200   NaN  
4                  NaN          200   NaN  


C:\Users\ziw857\AppData\Local\Temp\ipykernel_15172\553116324.py:16: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file, usecols=cols)
